In [1]:
!pip install torch torchvision sentencepiece tokenizers sacrebleu transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [42]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sacrebleu import corpus_bleu
from tqdm import tqdm
import sentencepiece as spm
import pandas as pd

In [43]:
with open('english-corpus.txt', 'r', encoding='utf-8') as f:
    english_sentences = f.read().splitlines()

with open('urdu-corpus.txt', 'r', encoding='utf-8') as f:
    urdu_sentences = f.read().splitlines()

# Ensure same length
assert len(english_sentences) == len(urdu_sentences)

df = pd.DataFrame({'en': english_sentences, 'ur': urdu_sentences})
train_df, test_df = train_test_split(df, test_size=0.1)


In [44]:
with open("train_en.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["en"]))

with open("train_ur.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["ur"]))

spm.SentencePieceTrainer.train(
    input='train_en.txt',
    model_prefix='en',
    vocab_size=3000,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)
spm.SentencePieceTrainer.train(
    input='train_ur.txt',
    model_prefix='ur',
    vocab_size=3000,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp_en = spm.SentencePieceProcessor(model_file='en.model')
sp_ur = spm.SentencePieceProcessor(model_file='ur.model')


In [45]:
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df["en"].tolist()
        self.tgt = df["ur"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_ids = [sp_en.bos_id()] + sp_en.encode(self.src[idx], out_type=int) + [sp_en.eos_id()]
        tgt_ids = [sp_ur.bos_id()] + sp_ur.encode(self.tgt[idx], out_type=int) + [sp_ur.eos_id()]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

def collate_fn(batch):
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=sp_en.pad_id())
    tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=sp_ur.pad_id())
    return src, tgt

train_data = TranslationDataset(train_df)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [46]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        self.encoder_embed = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.transformer = nn.Transformer(d_model, nhead, num_layers, num_layers)
        self.output = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        src_mask = self.transformer.generate_square_subsequent_mask(src.size(1)).to(src.device)
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)

        src = self.encoder_embed(src) * (self.d_model ** 0.5)
        tgt = self.decoder_embed(tgt) * (self.d_model ** 0.5)

        src = src.permute(1, 0, 2)  # (S, N, E)
        tgt = tgt.permute(1, 0, 2)

        out = self.transformer(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)
        out = out.permute(1, 0, 2)  # (N, T, E)
        return self.output(out)


In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerModel(8000, 8000).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=sp_ur.pad_id())

In [48]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.7)

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")


100%|██████████| 690/690 [00:33<00:00, 20.31it/s]


Epoch 1 Loss: 4.1586


100%|██████████| 690/690 [00:33<00:00, 20.42it/s]


Epoch 2 Loss: 2.9431


100%|██████████| 690/690 [00:34<00:00, 19.89it/s]


...

Epoch 8 Loss: 0.9976


100%|██████████| 690/690 [00:35<00:00, 19.68it/s]


Epoch 9 Loss: 0.8533


100%|██████████| 690/690 [00:34<00:00, 20.08it/s]

Epoch 10 Loss: 0.7319


In [49]:
def greedy_decode(model, sentence, max_len=50):
    model.eval()
    src_ids = [sp_en.bos_id()] + sp_en.encode(sentence, out_type=int) + [sp_en.eos_id()]
    src = torch.tensor(src_ids).unsqueeze(0).to(device)

    memory = model.encoder_embed(src) * (model.d_model ** 0.5)
    memory = memory.permute(1, 0, 2)

    tgt_ids = [sp_ur.bos_id()]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_ids).unsqueeze(0).to(device)
        tgt_emb = model.decoder_embed(tgt_tensor) * (model.d_model ** 0.5)
        tgt_emb = tgt_emb.permute(1, 0, 2)

        tgt_mask = model.transformer.generate_square_subsequent_mask(tgt_tensor.size(1)).to(device)
        output = model.transformer(memory, tgt_emb, tgt_mask=tgt_mask)
        output = output.permute(1, 0, 2)
        next_token_logits = model.output(output[:, -1, :])
        next_token = torch.argmax(next_token_logits, dim=-1).item()

        if next_token == sp_ur.eos_id():
            break
        tgt_ids.append(next_token)

    return sp_ur.decode(tgt_ids[1:])  # remove BOS token


In [50]:
preds, refs = [], []
for i in range(50):  # evaluate on 50 test samples
    en_sent = test_df.iloc[i]['en']
    ur_sent = test_df.iloc[i]['ur']

    translated = greedy_decode(model, en_sent)
    preds.append(translated)
    refs.append([ur_sent])

bleu = corpus_bleu(preds, refs)
print("BLEU Score:", bleu.score)

BLEU Score: 20.412414523193146


Additional Tasks

BYTE PAIR ENCODING

In [51]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

def train_bpe_tokenizer(lang_file, vocab_size, output_file):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
    )
    tokenizer.train([lang_file], trainer)
    tokenizer.save(output_file)
    return tokenizer

en_tokenizer = train_bpe_tokenizer("train_en.txt", 3000, "bpe_en.json")
ur_tokenizer = train_bpe_tokenizer("train_ur.txt", 3000, "bpe_ur.json")

Data Augmentation: Back Translation

In [52]:
from transformers import pipeline
from tqdm import tqdm

en2ur = pipeline("translation", model="facebook/mbart-large-50-many-to-many-mmt", tokenizer="facebook/mbart-large-50-many-to-many-mmt", src_lang="en_XX", tgt_lang="ur_PK")
ur2en = pipeline("translation", model="facebook/mbart-large-50-many-to-many-mmt", tokenizer="facebook/mbart-large-50-many-to-many-mmt", src_lang="ur_PK", tgt_lang="en_XX")

augmented_en, augmented_ur = [], []

with open("train_en.txt", "r", encoding="utf-8") as en_file:
    lines = en_file.readlines()

for line in tqdm(lines[:100]):  # You can increase this later
    try:
        urdu = en2ur(line)[0]["translation_text"]
        back_en = ur2en(urdu)[0]["translation_text"]
        augmented_en.append(back_en.strip())
        augmented_ur.append(urdu.strip())
    except Exception as e:
        continue

# Combine original and augmented
with open("final_en.txt", "w", encoding="utf-8") as f_en, open("final_ur.txt", "w", encoding="utf-8") as f_ur:
    with open("train_en.txt", "r") as orig_en, open("train_ur.txt", "r") as orig_ur:
        f_en.writelines(orig_en.readlines())
        f_ur.writelines(orig_ur.readlines())
    for e, u in zip(augmented_en, augmented_ur):
        f_en.write(e + "\n")
        f_ur.write(u + "\n")


Device set to use cuda:0
Device set to use cuda:0
100%|██████████| 100/100 [00:38<00:00,  2.57it/s]


In [53]:
import pandas as pd

# Read the English and Urdu sentences
with open("final_en.txt", "r", encoding="utf-8") as f_en, open("final_ur.txt", "r", encoding="utf-8") as f_ur:
    en_lines = [line.strip() for line in f_en.readlines()]
    ur_lines = [line.strip() for line in f_ur.readlines()]

# Check lengths
print(f"English: {len(en_lines)} lines")
print(f"Urdu: {len(ur_lines)} lines")

# Ensure equal length to avoid mismatch errors
min_len = min(len(en_lines), len(ur_lines))
en_lines = en_lines[:min_len]
ur_lines = ur_lines[:min_len]

# Create a DataFrame
df = pd.DataFrame({
    "en": en_lines,
    "ur": ur_lines
})

# Save to CSV
df.to_csv("final_data.csv", index=False)
print("✅ Saved final_data.csv!")


English: 22171 lines
Urdu: 22171 lines
✅ Saved final_data.csv!


In [56]:
import torch
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and tokenizer (make sure these are already defined)
model = model.to(device)

# Load the dataset
final_data = pd.read_csv("final_data.csv")

# Subset of the data (change 100 to 50 if needed)
subset = final_data.sample(n=100, random_state=42).reset_index(drop=True)

# Function to decode
def greedy_decode(model, sentence, max_len=50):
    model.eval()
    # Encode input sentence
    src_tokens = [sp_en.bos_id()] + sp_en.encode(sentence, out_type=int) + [sp_en.eos_id()]
    src_tensor = torch.tensor(src_tokens).unsqueeze(0).to(device)
    tgt_tensor = torch.tensor([[sp_ur.bos_id()]]).to(device)

    for _ in range(max_len):
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor)
            next_token = output.argmax(-1)[:, -1]  # last token
            tgt_tensor = torch.cat([tgt_tensor, next_token.unsqueeze(1)], dim=1)
            if next_token.item() == sp_ur.eos_id():
                break

    decoded = sp_ur.decode(tgt_tensor.squeeze(0).tolist()[1:-1])  # remove <bos> and <eos>
    return decoded


# Collect predictions and references
preds, refs = [], []

for i in range(len(subset)):
    en_sent = subset.iloc[i]['en']
    ur_sent = subset.iloc[i]['ur']

    try:
        translated = greedy_decode(model, en_sent)
        preds.append(translated)
        refs.append([ur_sent])
    except Exception as e:
        print(f"Error on index {i}: {e}")
        continue

# Compute BLEU score
bleu_score = corpus_bleu(refs, preds)
print("✅ BLEU Score on 100 samples:", bleu_score)


✅ BLEU Score on 100 samples: 0.8150143183638812


In [31]:
# from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainer, Seq2SeqTrainingArguments
# from datasets import Dataset
# import pandas as pd

# # Load tokenizer/model
# model_name = "facebook/mbart-large-50-many-to-many-mmt"
# tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
# model = MBartForConditionalGeneration.from_pretrained(model_name)

# tokenizer.src_lang = "en_XX"
# tokenizer.tgt_lang = "ur_PK"

# # Load your dataset
# data = pd.read_csv("final_data.csv")  # Let's create this below

# # Tokenize function
# def preprocess(example):
#     tokenized = tokenizer(
#         text=example["en"],
#         text_target=example["ur"],
#         max_length=128,
#         truncation=True,
#         padding="max_length"
#     )
#     # Add labels key for the target sequence
#     tokenized["labels"] = tokenized["input_ids"].copy()  # Copying input_ids as labels for training
#     return tokenized


# # Convert to Dataset
# from datasets import Dataset
# dataset = Dataset.from_pandas(data)
# dataset = dataset.map(preprocess, batched=False)

# # Training Args
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./mbart-finetuned-en-ur",
#     per_device_train_batch_size=1,
#     learning_rate=3e-5,
#     num_train_epochs=3,
#     save_total_limit=1,
#     evaluation_strategy="no",
#     predict_with_generate=True,
#     fp16=True,
# )

# # Trainer
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset,
#     tokenizer=tokenizer,
# )

# trainer.train()


Map:   0%|          | 0/22171 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-31-4051a16b7c24>:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 516.12 MiB is free. Process 2697 has 14.23 GiB memory in use. Of the allocated memory 13.94 GiB is allocated by PyTorch, and 159.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)